# Review of weeks 04 to 06
### 25 minutes · five steps on one real table

Five steps, five minutes each. Each cell says at the top what it wants. Fill in
the code, run it, move on. The five are one path, not five drills: the tail you
measure in step 1 is the tail that moves a metric in step 3, and the scaling you
look at in step 2 is what step 4 needs before a distance means anything.

The data is `dataset_custo_terapia_mama_PA2025.csv`: 3,616 outpatient events of
breast cancer therapy paid by private health plans in Pará in 2025. One row is
one visit. The columns used here are `custo_evento` (cost in reais),
`dose_total` (total dose billed), `n_medicamentos` (how many drugs were billed),
`n_itens_evento` (how many items) and `deslocamento` (1 if the patient was
treated outside her own city).

It covers meetings 04 to 07: preprocessing, supervised learning, clustering and
trees.

On Colab, drag the CSV into the file browser on the left before you start.

Open `review_weeks_04_06_answers.ipynb` only after you are done. A correct
answer you have read is not an answer you can write.

In [ ]:
# TASK
# Find the upper end of custo_evento with the interquartile range.
#   (a) compute Q1, Q3 and IQR, then lower_bound and upper_bound with the
#       1.5 rule, and print the five numbers
#   (b) print how many events sit ABOVE the upper bound, what share of the
#       rows that is, and what share of all the money in the file they carry
#   (c) in FILTER_DOES, say in one sentence what a filter that keeps only the
#       rows between the two bounds does to those events: does it take them
#       out, or does it put another value in their place?

import numpy as np
import pandas as pd


df = pd.read_csv("dataset_custo_terapia_mama_PA2025.csv")

# (a) compute Q1, Q3 and IQR, then lower_bound and upper_bound with the 1.5 rule
q1 = df["custo_evento"].quantile(0.25)
q3 = df["custo_evento"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}, Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")

# (b) print how many events sit ABOVE the upper bound, what share of the
#     rows that is, and what share of all the money in the file they carry
events_above_upper_bound = df[df["custo_evento"] > upper_bound].shape[0]
total_events = df.shape[0]
share_of_rows = (events_above_upper_bound / total_events) * 100

total_money_above_upper_bound = df[df["custo_evento"] > upper_bound]["custo_evento"].sum()
total_money_in_file = df["custo_evento"].sum()
share_of_money = (total_money_above_upper_bound / total_money_in_file) * 100

print(f"\nEvents above upper bound: {events_above_upper_bound}")
print(f"Share of rows above upper bound: {share_of_rows:.2f}%")
print(f"Share of total money carried by these events: {share_of_money:.2f}%")

# (c) apply the filter itself: keep only the rows between the two bounds
df_filtered = df[(df["custo_evento"] >= lower_bound) & (df["custo_evento"] <= upper_bound)]
print(f"\nRows before filter: {df.shape[0]}, rows after filter: {df_filtered.shape[0]}")

FILTER_DOES = "A filter that keeps only the rows between the two bounds takes out the events outside those bounds, it does not replace their values."

print(f"\nFilter explanation: {FILTER_DOES}")

---
## 2. Four columns, four scales · 5 minutes

Step 4 will ask K-means for groups, and K-means measures distance. Before a
distance can mean anything, look at what these columns actually range over.

In [ ]:
# TASK
# Run the given lines first, then:
#   (a) fill GUARANTEES: what does each transformation guarantee about every
#       column after it runs?
#   (b) fill SAME_NUMBERS with True or False: do the two produce the same
#       transformed table?
#   (c) in WHY_SCALE, say in one sentence why a K-means on the RAW columns
#       would be decided almost entirely by custo_evento

from sklearn.preprocessing import MinMaxScaler, StandardScaler

NUMERIC = ["custo_evento", "dose_total", "n_medicamentos", "n_itens_evento"]
print(df[NUMERIC].agg(["min", "max", "std"]).round(2))

minmax = MinMaxScaler().fit_transform(df[NUMERIC])
standard = StandardScaler().fit_transform(df[NUMERIC])

print("\nmin-max  -> min", minmax.min(axis=0).round(2),
      "max", minmax.max(axis=0).round(2))
print("standard -> mean", standard.mean(axis=0).round(2),
      "std", standard.std(axis=0).round(2))

GUARANTEES = {
    "MinMaxScaler": "Squeezes every column into the [0, 1] range (min becomes 0, max becomes 1), but does not control the mean or the std.",
    "StandardScaler": "Centers every column to mean 0 and scales it to std 1, but does not bound it to a fixed min/max range.",
}

SAME_NUMBERS = False

WHY_SCALE = "custo_evento has a much larger raw range and std than the other columns, so on the raw scale its differences dominate the Euclidean distance and the other columns barely move it."

print(GUARANTEES)
print("same numbers?", SAME_NUMBERS)
print(WHY_SCALE)


---
## 3. Two numbers about one model · 5 minutes

Now fit something and read it. Two metrics, and they do not answer the same
question — that is the whole exercise.

In [ ]:
# TASK
# Write this one yourself. The baselines at the bottom are given; everything
# above them is yours.
#   (a) build X from FEATURES and y from custo_evento
#   (b) split them, 25% for test, random_state=42
#   (c) fit a LinearRegression on the TRAINING data only and predict on the test
#   (d) print the MAE and the R2 of those predictions
#   (e) run the given baselines, then fill READING: what does a negative R2
#       mean, and why does the median baseline beat the mean baseline on MAE
#       while losing to it on R2?

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

FEATURES = ["dose_total", "n_medicamentos", "n_itens_evento", "deslocamento"]

# (a) to (d), your answer here
X = df[FEATURES]
y = df["custo_evento"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"LinearRegression -> MAE = {mae:11,.2f}   R2 = {r2:8.4f}")


# (e) given: two models that ignore the features and always answer the same
for name, constant in (("mean", y_train.mean()), ("median", y_train.median())):
    guess = np.full(len(y_test), constant)
    print(f"{name:8s} always says {constant:11,.2f}   "
          f"MAE = {mean_absolute_error(y_test, guess):11,.2f}   "
          f"R2 = {r2_score(y_test, guess):8.4f}")

READING = ("A negative R2 means the model does worse than simply always guessing the "
           "training mean: R2 = 0 is exactly the mean baseline's score, so anything "
           "below zero is a model that adds error instead of removing it. The median "
           "wins on MAE because MAE is minimized by the median (it only cares about "
           "the middle of the distribution), while the mean wins on R2 because R2 is "
           "built from squared error, which is minimized by the mean, not the median.")

print(READING)


---
## 4. How many groups · 5 minutes

Step 2 put the columns on a common scale. Use them, and let two pieces of
evidence argue about k before you commit to one.

In [ ]:
# TASK
# Run the given scaling, then:
#   (a) fill the loop so it prints, for each k, the inertia and the silhouette
#   (b) fill BEST_K with the k you would defend and WHY with the evidence
#   (c) fill `labels`: fit a KMeans with BEST_K and get the cluster of every
#       row in a SINGLE call

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

CLUSTERING = ["custo_evento", "dose_total", "n_itens_evento"]
X_scaled = StandardScaler().fit_transform(df[CLUSTERING])

for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia = km.inertia_
    silhouette = silhouette_score(X_scaled, km.labels_)
    print(f"k = {k}   inertia = {inertia:9,.1f}   silhouette = {silhouette:.3f}")

BEST_K = 4
WHY = ("k=4 has the highest silhouette score of the five tried (0.726, clearly above "
       "its neighbours 3 and 5), and it also sits where the inertia curve's drop "
       "slows down noticeably (the elbow) — both pieces of evidence point to the "
       "same k.")

labels = KMeans(n_clusters=BEST_K, random_state=42, n_init=10).fit_predict(X_scaled)

print("chosen k:", BEST_K, "-", WHY)
print(df.assign(cluster=labels)
        .groupby("cluster")["custo_evento"].agg(["size", "mean"]).round(2))


---
## 5. The whole flow, from memory · 5 minutes

Nothing is given this time. Four lines carry every supervised model you will
ever fit; write them without looking.

In [ ]:
# TASK
# Write the whole thing yourself, nothing is given.
#   (a) build X from FEATURES and y from custo_evento, then split them with
#       25% for test and random_state=42
#   (b) fit a DecisionTreeRegressor(random_state=42) on the training data and
#       predict on the test data
#   (c) print the MAE and the R2 of those predictions
#   (d) fit a RandomForestRegressor(n_estimators=100, random_state=42) on the
#       same split, print its MAE and R2, and compare

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

X = df[FEATURES]
y = df["custo_evento"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
print(f"DecisionTree  -> MAE = {mean_absolute_error(y_test, tree_pred):11,.2f}   "
      f"R2 = {r2_score(y_test, tree_pred):8.4f}")

forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)
print(f"RandomForest  -> MAE = {mean_absolute_error(y_test, forest_pred):11,.2f}   "
      f"R2 = {r2_score(y_test, forest_pred):8.4f}")

# the forest beats the single tree on both metrics here: averaging many trees
# trained on bootstrapped samples smooths out the single tree's overfitting.


---
## Before you close this

| If this step went badly | Revise |
|---|---|
| 1 | quartiles, the 1.5 IQR rule, and removing against replacing |
| 2 | Min-Max against standardisation, and why a distance needs one scale |
| 3 | train/test split, MAE, R2, and what a negative R2 means |
| 4 | inertia and the elbow, silhouette, and `fit_predict` |
| 5 | `fit`, `predict`, and the argument order of a metric |

Steps 3 and 5 are the ones that reward having typed the code rather than having
read it. If you only have time to redo one thing, redo those two on paper,
without the notebook open.